In [2]:
import pandas as pd
import numpy as np
import json
import os
import logging
import sys

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', stream=sys.stdout)

# --- Configuration for Week 13 (Aggregating Round 11 Data) ---

# Previous master file (contains data up to Round 10)
MASTER_FILE_PATH_OLD = 'bbo_master_w12.csv'

# New master file for Week 13 (will contain data up to Round 11)
MASTER_FILE_PATH_NEW = 'bbo_master_w13.csv'

ADD_DATA_DIR = 'add_data'

# Files for the new data point (Round 11 data)
INPUTS_FILE = 'week12_clean_inputs.json'   # Generated by your w12 capstone script
OUTPUTS_FILE = 'week12_clean_outputs.json'  # The new scores you just received

NUM_FUNCTIONS = 8
CURRENT_ROUND = 11  # Appending Round 11 data

# Function Dimensions
FUNCTION_DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}

def load_json_file(file_path):
    full_path = os.path.join(ADD_DATA_DIR, file_path)
    if not os.path.exists(full_path):
        logging.error(f"ERROR: {full_path} not found.")
        return None
    with open(full_path, 'r') as f:
        return json.load(f)

def create_master_data():
    logging.info("*"*50)
    logging.info(f"--- Starting BBO Master File Creation for Round {CURRENT_ROUND} ---")

    # 1. Load Old Master
    if not os.path.exists(MASTER_FILE_PATH_OLD):
        logging.error(f"Master file {MASTER_FILE_PATH_OLD} not found.")
        return

    df_old = pd.read_csv(MASTER_FILE_PATH_OLD)
    logging.info(f"Loaded {len(df_old)} rows from {MASTER_FILE_PATH_OLD}.")

    # 2. Load New Data
    # Note: If output is .txt, we handle it as json if it's formatted as a list
    inputs = load_json_file(INPUTS_FILE)
    outputs = load_json_file(OUTPUTS_FILE)

    if not inputs or not outputs:
        logging.error("Missing input/output data.")
        return

    # 3. Construct New Rows
    new_rows = []
    cols = df_old.columns.tolist()

    for i in range(NUM_FUNCTIONS):
        f_id = i + 1
        dim = FUNCTION_DIMS[f_id]
        
        # Get coords and score
        coords = inputs[i]
        score = outputs[i]

        # Truncate or Pad coords to 8 dimensions
        # Truncate if too long (e.g. from a copy-paste error)
        if len(coords) > dim:
            coords = coords[:dim]
        # Pad with NaNs for CSV storage
        coords_padded = coords + [np.nan] * (8 - len(coords))

        row = {
            'Function ID': f_id,
            'Round': CURRENT_ROUND,
            'Y': score
        }
        # Add X values
        for d in range(8):
            row[f'X{d+1}'] = coords_padded[d]

        new_rows.append(row)

    df_new = pd.DataFrame(new_rows, columns=cols)

    # 4. Concatenate and Save
    df_final = pd.concat([df_old, df_new], ignore_index=True)
    
    # Ensure 'Round' column is integer
    if 'Round' in df_final.columns:
        df_final['Round'] = df_final['Round'].fillna(0).astype(int)

    df_final.to_csv(MASTER_FILE_PATH_NEW, index=False)

    logging.info(f"SUCCESS: {MASTER_FILE_PATH_NEW} created.")
    logging.info(f"Total rows: {len(df_final)} (Expected 168 + 8 = 176).")
    logging.info(f"Verification: {len(df_final)/8} data points per function.")
    logging.info("*"*50)
    
    # Print tail to verify
    print("\n--- New Data Added ---")
    print(df_final[df_final['Round'] == CURRENT_ROUND][['Function ID', 'Y']])

if __name__ == '__main__':
    create_master_data()


INFO: **************************************************
INFO: --- Starting BBO Master File Creation for Round 11 ---
INFO: Loaded 168 rows from bbo_master_w12.csv.
INFO: SUCCESS: bbo_master_w13.csv created.
INFO: Total rows: 176 (Expected 168 + 8 = 176).
INFO: Verification: 22.0 data points per function.
INFO: **************************************************

--- New Data Added ---
     Function ID              Y
160            1   1.700777e-15
161            2   7.281983e-01
162            3  -1.013003e-02
163            4  -1.845638e+00
164            5   8.662405e+03
165            6  -4.511237e-01
166            7   4.174490e-01
167            8   3.562991e+00
168            1  1.083313e-138
169            2   6.117367e-01
170            3  -8.763621e-02
171            4   1.436199e-01
172            5   8.662405e+03
173            6  -4.162279e-01
174            7   6.498129e-01
175            8   4.109311e+00
